In [ ]:
# satis yaptiktan sonra satis fiyatlarini gorme
from ib_insync import *
import random

# Notebook'ta çoğu zaman gerekmez ama bazı ortamlarda yardımcı olur
try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass

ib = IB()

# Eski bağlantı varsa kapat
if ib.isConnected():
    ib.disconnect()

# Benzersiz clientId kullan
client_id = random.randint(1000, 9999)
print("clientId:", client_id)

# TWS paper için genelde 7497, live için çoğu zaman 7496
await ib.connectAsync('127.0.0.1', 7497, clientId=client_id, timeout=10)

# Execution filtrele
exec_filter = ExecutionFilter(
    symbol='ISRG',
    secType='STK',
    side='SELL'   # burada SELL kullanılmalı
)

fills = await ib.reqExecutionsAsync(exec_filter)

if not fills:
    print("ISRG için satış execution bulunamadı.")
else:
    # en son execution
    last_fill = fills[-1]

    print("En son satış execution:")
    print("zaman     :", last_fill.time)
    print("symbol    :", last_fill.contract.symbol)
    print("side      :", last_fill.execution.side)      # burada genelde SLD döner
    print("adet      :", last_fill.execution.shares)
    print("fiyat     :", last_fill.execution.price)
    print("avgPrice  :", last_fill.execution.avgPrice)
    print("orderId   :", last_fill.execution.orderId)
    print("permId    :", last_fill.execution.permId)

clientId: 5608
En son satış execution:
zaman     : 2026-04-23 18:24:37+00:00
symbol    : ISRG
side      : SLD
adet      : 1.0
fiyat     : 481.71
avgPrice  : 481.71
orderId   : 0
permId    : 689589580


In [ ]:
#market buy only
from ib_insync import *
import asyncio
import random

ib = IB()

if ib.isConnected():
    ib.disconnect()

await ib.connectAsync(
    '127.0.0.1',
    7497,   # paper TWS ise genelde bu
    clientId=random.randint(1000, 9999),
    timeout=10
)

contract = Stock('NVDA', 'SMART', 'USD')
contract = (await ib.qualifyContractsAsync(contract))[0]

order = MarketOrder('BUY', 1)
order.tif = 'DAY'
order.transmit = True

trade = ib.placeOrder(contract, order)

while not trade.isDone():
    await asyncio.sleep(0.5)

if trade.fills:
    last_fill = trade.fills[-1]
    print("time     :", last_fill.time)
    print("symbol   :", last_fill.contract.symbol)
    print("side     :", last_fill.execution.side)
    print("shares   :", last_fill.execution.shares)
    print("price    :", last_fill.execution.price)
    print("avgPrice :", last_fill.execution.avgPrice)
else:
    print("Fill yok")
    print("status:", trade.orderStatus.status)
    print("log:")
    for row in trade.log:
        print(row.time, row.status, row.message, row.errorCode)

time     : 2026-04-23 18:53:05.226020+00:00
symbol   : NVDA
side     : BOT
shares   : 1.0
price    : 198.9
avgPrice : 198.9


## market buy + trailing stop emri %2

In [6]:
# buy market yapar, ama hemen trailing stop emri de  verir. %2 burada verdik
from ib_insync import *
import asyncio
import random

ib = IB()

# varsa eski bağlantıyı kapat
if ib.isConnected():
    ib.disconnect()

# bağlan
await ib.connectAsync(
    '127.0.0.1',
    7497,  # paper TWS ise genelde 7497
    clientId=random.randint(1000, 9999),
    timeout=10
)

# kontrat
contract = Stock('GRAB', 'SMART', 'USD')
contract = (await ib.qualifyContractsAsync(contract))[0]

# 1) market buy
buy_order = MarketOrder('BUY', 1)
buy_order.tif = 'DAY'
buy_order.transmit = True

buy_trade = ib.placeOrder(contract, buy_order)

while not buy_trade.isDone():
    await asyncio.sleep(0.5)

print("BUY status:", buy_trade.orderStatus.status)

if not buy_trade.fills:
    print("Alış fill gelmedi.")
    print("BUY LOG:")
    for row in buy_trade.log:
        print(row.time, row.status, row.message, row.errorCode)
else:
    last_buy_fill = buy_trade.fills[-1]
    filled_qty = int(last_buy_fill.execution.shares)
    buy_price = last_buy_fill.execution.price

    print("\nMSFT alış detayları:")
    print("time     :", last_buy_fill.time)
    print("symbol   :", last_buy_fill.contract.symbol)
    print("side     :", last_buy_fill.execution.side)
    print("shares   :", last_buy_fill.execution.shares)
    print("price    :", buy_price)
    print("avgPrice :", last_buy_fill.execution.avgPrice)

    # 2) trailing stop sell (%2)
    trail_order = Order()
    trail_order.action = 'SELL'
    trail_order.orderType = 'TRAIL'
    trail_order.totalQuantity = filled_qty
    trail_order.trailingPercent = 2.0
    trail_order.tif = 'DAY'
    trail_order.transmit = True

    # İstersen başlangıç stop seviyesini de açıkça verebilirsin:
    # trail_order.trailStopPrice = round(buy_price * 0.98, 2)

    trail_trade = ib.placeOrder(contract, trail_order)

    await asyncio.sleep(1.0)

    print("\nTrailing stop emri gönderildi:")
    print("status   :", trail_trade.orderStatus.status)
    print("action   :", trail_trade.order.action)
    print("type     :", trail_trade.order.orderType)
    print("qty      :", trail_trade.order.totalQuantity)
    print("trail %  :", trail_trade.order.trailingPercent)

    if trail_trade.log:
        print("\nTRAIL LOG:")
        for row in trail_trade.log:
            print(row.time, row.status, row.message, row.errorCode)

BUY status: Filled

MSFT alış detayları:
time     : 2026-04-23 18:56:32.451324+00:00
symbol   : GRAB
side     : BOT
shares   : 1.0
price    : 3.95
avgPrice : 3.95

Trailing stop emri gönderildi:
status   : PreSubmitted
action   : SELL
type     : TRAIL
qty      : 1.0
trail %  : 2.0

TRAIL LOG:
2026-04-23 18:56:32.797353+00:00 PendingSubmit  0
2026-04-23 18:56:32.830061+00:00 PreSubmitted  0


# market buy emri + exit modu. butun modlar var sen sadece hangi cikis modunu istedigini parametre olarak veriyorsun. 

In [9]:

from ib_insync import *
import asyncio
import random

# ------------------------------------------------------------
# EXIT TYPE MAPPING
# ------------------------------------------------------------
EXIT_ORDER_TYPES = {
    # Hemen piyasadan çık
    "market": {
        "orderType": "MKT",
        "required": [],
        "optional": ["tif", "outsideRth"]
    },

    # Belirli fiyattan ya da daha iyisinden çık
    "limit": {
        "orderType": "LMT",
        "required": ["limit_price"],
        "optional": ["tif", "outsideRth"]
    },

    # Fiyat stop seviyesine gelirse market satış
    "stop": {
        "orderType": "STP",
        "required": ["stop_price"],
        "optional": ["tif", "outsideRth"]
    },

    # Fiyat stop seviyesine gelirse limit satış
    "stop_limit": {
        "orderType": "STP LMT",
        "required": ["stop_price", "limit_price"],
        "optional": ["tif", "outsideRth"]
    },

    # Fiyat belirli seviyeye değerse market satış
    "market_if_touched": {
        "orderType": "MIT",
        "required": ["trigger_price"],
        "optional": ["tif", "outsideRth"]
    },

    # Trailing stop - sabit tutar ile
    "trailing_stop_amount": {
        "orderType": "TRAIL",
        "required": ["trail_amount"],
        "optional": ["trail_stop_price", "tif", "outsideRth"]
    },

    # Trailing stop - yüzde ile
    "trailing_stop_percent": {
        "orderType": "TRAIL",
        "required": ["trailing_percent"],
        "optional": ["trail_stop_price", "tif", "outsideRth"]
    },

    # Kapanışta marketten çık
    "market_on_close": {
        "orderType": "MOC",
        "required": [],
        "optional": []
    },

    # Açılışta marketten çık
    "market_on_open": {
        "orderType": "MKT",
        "required": [],
        "optional": ["outsideRth"],
        "forced_tif": "OPG"
    },
}


# ------------------------------------------------------------
# EXIT ORDER BUILDER
# ------------------------------------------------------------
def build_exit_order(exit_type: str, qty: float, **params) -> Order:
    if exit_type not in EXIT_ORDER_TYPES:
        raise ValueError(
            f"Geçersiz exit_type: {exit_type}. "
            f"Geçerli tipler: {list(EXIT_ORDER_TYPES.keys())}"
        )

    spec = EXIT_ORDER_TYPES[exit_type]

    missing = [p for p in spec["required"] if p not in params]
    if missing:
        raise ValueError(
            f"{exit_type} için eksik parametre(ler): {missing}"
        )

    o = Order()
    o.action = "SELL"
    o.orderType = spec["orderType"]
    o.totalQuantity = qty
    o.transmit = True

    if "forced_tif" in spec:
        o.tif = spec["forced_tif"]
    else:
        o.tif = params.get("tif", "DAY")

    if "outsideRth" in params:
        o.outsideRth = params["outsideRth"]

    if exit_type == "market":
        pass

    elif exit_type == "limit":
        o.lmtPrice = float(params["limit_price"])

    elif exit_type == "stop":
        # IBKR'de stop trigger fiyatı auxPrice ile verilir
        o.auxPrice = float(params["stop_price"])

    elif exit_type == "stop_limit":
        o.auxPrice = float(params["stop_price"])
        o.lmtPrice = float(params["limit_price"])

    elif exit_type == "market_if_touched":
        # IBKR'de MIT trigger fiyatı auxPrice ile verilir
        o.auxPrice = float(params["trigger_price"])

    elif exit_type == "trailing_stop_amount":
        # IBKR'de TRAIL order için sabit trailing amount auxPrice ile verilir
        o.auxPrice = float(params["trail_amount"])
        if "trail_stop_price" in params:
            o.trailStopPrice = float(params["trail_stop_price"])

    elif exit_type == "trailing_stop_percent":
        o.trailingPercent = float(params["trailing_percent"])
        if "trail_stop_price" in params:
            o.trailStopPrice = float(params["trail_stop_price"])

    elif exit_type == "market_on_close":
        pass

    elif exit_type == "market_on_open":
        pass

    return o


# ------------------------------------------------------------
# BUY WITH EXIT
# ------------------------------------------------------------
async def buy_with_exit(ib: IB, symbol: str, qty: int, exit_type: str, **exit_params):
    """
    Market BUY yapar, fill geldikten sonra seçilen exit order'ı gönderir.
    """

    contract = Stock(symbol, 'SMART', 'USD')
    contract = (await ib.qualifyContractsAsync(contract))[0]

    # 1) MARKET BUY
    buy_order = MarketOrder('BUY', qty)
    buy_order.tif = 'DAY'
    buy_order.transmit = True

    buy_trade = ib.placeOrder(contract, buy_order)

    while not buy_trade.isDone():
        await asyncio.sleep(0.5)

    if not buy_trade.fills:
        return {
            "ok": False,
            "reason": "Buy fill gelmedi",
            "buy_status": buy_trade.orderStatus.status,
            "buy_log": [
                {
                    "time": str(x.time),
                    "status": x.status,
                    "message": x.message,
                    "errorCode": x.errorCode
                }
                for x in buy_trade.log
            ]
        }

    last_fill = buy_trade.fills[-1]
    buy_time = str(last_fill.time)
    buy_symbol = last_fill.contract.symbol
    buy_shares = last_fill.execution.shares
    buy_price = last_fill.execution.price
    buy_avg_price = last_fill.execution.avgPrice

    # 2) EXIT ORDER
    exit_order = build_exit_order(
        exit_type=exit_type,
        qty=buy_shares,
        **exit_params
    )

    exit_trade = ib.placeOrder(contract, exit_order)

    await asyncio.sleep(1.0)

    # rapor için exit detayları
    exit_detail = {}
    if exit_type == "trailing_stop_amount":
        exit_detail["trail_amount"] = exit_params.get("trail_amount")
        exit_detail["trail_stop_price"] = exit_params.get("trail_stop_price")

    elif exit_type == "trailing_stop_percent":
        exit_detail["trailing_percent"] = exit_params.get("trailing_percent")
        exit_detail["trail_stop_price"] = exit_params.get("trail_stop_price")

    elif exit_type == "stop":
        exit_detail["stop_price"] = exit_params.get("stop_price")

    elif exit_type == "stop_limit":
        exit_detail["stop_price"] = exit_params.get("stop_price")
        exit_detail["limit_price"] = exit_params.get("limit_price")

    elif exit_type == "limit":
        exit_detail["limit_price"] = exit_params.get("limit_price")

    elif exit_type == "market_if_touched":
        exit_detail["trigger_price"] = exit_params.get("trigger_price")

    return {
        "ok": True,
        "symbol": buy_symbol,
        "buy_time": buy_time,
        "buy_shares": buy_shares,
        "buy_price": buy_price,
        "buy_avg_price": buy_avg_price,
        "exit_type": exit_type,
        "exit_order_status": exit_trade.orderStatus.status,
        "exit_order_type": exit_trade.order.orderType,
        "exit_detail": exit_detail
    }

In [ ]:
result = await buy_with_exit(
    ib,
    symbol='MSFT',
    qty=1,
    exit_type='trailing_stop_amount',
    trail_amount=5.0
)

print(result)

# ornek bi target price var ise o zaman bu alltaki gibi olacak

# result = await buy_with_exit(
#     ib,
#     symbol='MSFT',
#     qty=1,
#     exit_type='limit',
#     limit_price=200 # bizim belirledigimiz target price
# )

{'ok': True, 'symbol': 'MSFT', 'buy_time': '2026-04-23 19:16:45.504925+00:00', 'buy_shares': 1.0, 'buy_price': 416.67, 'buy_avg_price': 416.67, 'exit_type': 'trailing_stop_amount', 'exit_order_status': 'PreSubmitted', 'exit_order_type': 'TRAIL', 'exit_detail': {'trail_amount': 5.0, 'trail_stop_price': None}}


# market sell emri, aninda satar ve satis fiyatini ve kar yuzdesini gosterir

In [18]:
from ib_insync import *
import asyncio

async def get_weighted_avg_buy_price_from_fills(ib: IB, symbol: str):
    """
    Verilen sembol için BUY execution'lardan ağırlıklı ortalama alış fiyatını hesaplar.
    """

    contract_symbol = symbol.upper()

    fills = await ib.reqExecutionsAsync()

    buy_fills = []
    for f in fills:
        if f.contract.symbol == contract_symbol and f.execution.side in ('BOT', 'BUY'):
            buy_fills.append(f)

    if not buy_fills:
        return None

    total_qty = 0.0
    total_amount = 0.0

    for f in buy_fills:
        qty = float(f.execution.shares)
        px = float(f.execution.price)
        total_qty += qty
        total_amount += qty * px

    if total_qty == 0:
        return None

    return total_amount / total_qty


async def market_sell_with_true_avg_buy_price(ib: IB, symbol: str):
    """
    1) Açık pozisyonu bulur
    2) BUY fill'lerden ağırlıklı ortalama alış fiyatını hesaplar
    3) Market sell yapar
    4) Gerçek filled satış fiyatı, zamanı ve %PnL döner
    """

    symbol = symbol.upper()

    # Açık pozisyonu bul
    positions = ib.positions()
    pos = None
    for p in positions:
        if p.contract.symbol == symbol and p.position > 0:
            pos = p
            break

    if pos is None:
        return {
            "ok": False,
            "reason": f"{symbol} için açık long pozisyon bulunamadı"
        }

    qty = float(pos.position)
    contract = pos.contract

    # BUY fill'lerden gerçek ortalama alış fiyatı
    buy_price = await get_weighted_avg_buy_price_from_fills(ib, symbol)

    if buy_price is None:
        return {
            "ok": False,
            "reason": f"{symbol} için BUY execution bulunamadı"
        }

    # Market sell
    sell_order = MarketOrder('SELL', qty)
    sell_order.tif = 'DAY'
    sell_order.transmit = True

    trade = ib.placeOrder(contract, sell_order)

    while not trade.isDone():
        await asyncio.sleep(0.5)

    if not trade.fills:
        return {
            "ok": False,
            "reason": "Sell fill gelmedi",
            "status": trade.orderStatus.status,
            "log": [
                {
                    "time": str(x.time),
                    "status": x.status,
                    "message": x.message,
                    "errorCode": x.errorCode
                }
                for x in trade.log
            ]
        }

    last_fill = trade.fills[-1]

    sell_price = float(last_fill.execution.price)
    sell_time = str(last_fill.time)
    filled_qty = float(last_fill.execution.shares)

    pct_change = ((sell_price - buy_price) / buy_price) * 100.0

    return {
        "ok": True,
        "symbol": symbol,
        "qty": filled_qty,
        "buy_price": round(buy_price, 4),
        "sell_price": round(sell_price, 4),
        "sell_time": sell_time,
        "pct_change": round(pct_change, 4),
        "status": trade.orderStatus.status
    }

In [21]:
result = await market_sell_with_true_avg_buy_price(ib, 'AAPL')
print(result)

{'ok': True, 'symbol': 'AAPL', 'qty': 1.0, 'buy_price': 273.9514, 'sell_price': 274.14, 'sell_time': '2026-04-23 19:30:00.706685+00:00', 'pct_change': 0.0688, 'status': 'Filled'}


# alis fiyati elle belirlenir + satis fiyati da elle belirlenir
ornek: entry price alis icin emir verir. satis icinde hede fiyata ulasinca sat emri verir

In [22]:
from ib_insync import *
import asyncio
import random

async def place_entry_and_target_order(
    ib: IB,
    symbol: str,
    qty: int,
    entry_price: float,
    target_price: float
):
    """
    entry_price'a gelirse BUY LMT ile alır,
    alım fill olursa target_price'da SELL LMT koyar.
    """

    contract = Stock(symbol, 'SMART', 'USD')
    contract = (await ib.qualifyContractsAsync(contract))[0]

    # Parent: entry buy limit
    parent = LimitOrder('BUY', qty, entry_price)
    parent.tif = 'DAY'
    parent.transmit = False

    # Önce parent'ı gönderip orderId alalım
    parent_trade = ib.placeOrder(contract, parent)

    # küçük bekleme: orderId dolsun
    while parent.orderId == 0:
        await asyncio.sleep(0.1)

    # Child: target sell limit
    child = LimitOrder('SELL', qty, target_price)
    child.parentId = parent.orderId
    child.tif = 'DAY'
    child.transmit = True

    child_trade = ib.placeOrder(contract, child)

    return {
        "ok": True,
        "symbol": symbol,
        "qty": qty,
        "entry_price": entry_price,
        "target_price": target_price,
        "parent_order_id": parent.orderId,
        "child_parent_id": child.parentId,
        "entry_status": parent_trade.orderStatus.status,
        "target_status": child_trade.orderStatus.status,
    }

In [23]:
result = await place_entry_and_target_order(
    ib,
    symbol='CRM',
    qty=1,
    entry_price=190.00,
    target_price=200.00
)

print(result)

{'ok': True, 'symbol': 'CRM', 'qty': 1, 'entry_price': 190.0, 'target_price': 200.0, 'parent_order_id': 18, 'child_parent_id': 18, 'entry_status': 'PendingSubmit', 'target_status': 'PendingSubmit'}


# entry price manual ama cikis icin trailing stop loss koyarak emir verme

In [24]:
from ib_insync import *
import asyncio
import random

async def place_entry_with_trailing_stop(
    ib: IB,
    symbol: str,
    qty: int,
    entry_price: float,
    trailing_percent: float = 2.0
):
    """
    entry_price'a gelirse BUY LIMIT ile alır.
    Parent fill olduktan sonra otomatik SELL TRAIL (% trailing stop) aktif olur.
    """

    contract = Stock(symbol, 'SMART', 'USD')
    contract = (await ib.qualifyContractsAsync(contract))[0]

    # --------------------------------------------------
    # 1) Parent order: BUY LIMIT
    # --------------------------------------------------
    parent = LimitOrder('BUY', qty, entry_price)
    parent.tif = 'DAY'
    parent.transmit = False

    parent_trade = ib.placeOrder(contract, parent)

    # parent orderId oluşana kadar bekle
    while parent.orderId == 0:
        await asyncio.sleep(0.1)

    # --------------------------------------------------
    # 2) Child order: SELL TRAIL
    # --------------------------------------------------
    child = Order()
    child.action = 'SELL'
    child.orderType = 'TRAIL'
    child.totalQuantity = qty
    child.trailingPercent = trailing_percent
    child.parentId = parent.orderId
    child.tif = 'DAY'
    child.transmit = True

    child_trade = ib.placeOrder(contract, child)

    return {
        "ok": True,
        "symbol": symbol,
        "qty": qty,
        "entry_price": entry_price,
        "trailing_percent": trailing_percent,
        "parent_order_type": parent.orderType,
        "child_order_type": child.orderType,
        "entry_status": parent_trade.orderStatus.status,
        "exit_status": child_trade.orderStatus.status,
        "parent_order_id": parent.orderId,
        "child_parent_id": child.parentId
    }

In [25]:
result = await place_entry_with_trailing_stop(
    ib,
    symbol='GIB',
    qty=1,
    entry_price=80.0,
    trailing_percent=2.0
)

print(result)

{'ok': True, 'symbol': 'GIB', 'qty': 1, 'entry_price': 80.0, 'trailing_percent': 2.0, 'parent_order_type': 'LMT', 'child_order_type': 'TRAIL', 'entry_status': 'PendingSubmit', 'exit_status': 'PendingSubmit', 'parent_order_id': 21, 'child_parent_id': 21}


# satin almadan once symbolun alim fiyatini ogrenme

In [53]:
import os
import certifi
import logging
logging.getLogger('tvDatafeed').setLevel(logging.CRITICAL)

from tvDatafeed import TvDatafeed, Interval

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

from tvDatafeed import TvDatafeed, Interval

tv = TvDatafeed()

df = tv.get_hist(
    symbol="NVDA",
    exchange="NASDAQ",
    interval=Interval.in_1_minute,
    n_bars=10
)

print("last price:", float(df["close"].iloc[-1]))
df

last price: 199.59


,symbol,open,high,low,close,volume
datetime,,,,,,
2026-04-23 21:50:00,NASDAQ:NVDA,199.450,199.47,198.710,199.10,78454.0
2026-04-23 21:51:00,NASDAQ:NVDA,199.100,199.15,198.910,199.11,33702.0
2026-04-23 21:52:00,NASDAQ:NVDA,199.110,199.32,199.070,199.31,28851.0
2026-04-23 21:53:00,NASDAQ:NVDA,199.300,199.47,199.300,199.39,30558.0
2026-04-23 21:54:00,NASDAQ:NVDA,199.380,199.45,199.280,199.42,28648.0
2026-04-23 21:55:00,NASDAQ:NVDA,199.420,199.51,199.320,199.48,47201.0
2026-04-23 21:56:00,NASDAQ:NVDA,199.495,199.64,199.450,199.59,27247.0
2026-04-23 21:57:00,NASDAQ:NVDA,199.595,199.62,199.480,199.60,29889.0
2026-04-23 21:58:00,NASDAQ:NVDA,199.590,199.70,199.575,199.65,43756.0


In [51]:
# pip install --upgrade --no-cache-dir git+https://github.com/rongardF/tvdatafeed.git
# pip install tradingview-datafeed
# pip install -U certifi

# portfoy bilgisi

In [63]:
from ib_insync import *
import pandas as pd
from collections import defaultdict, deque
import asyncio


# --------------------------------------------------
# 1) EXECUTIONS ÇEK
# --------------------------------------------------
async def get_all_ibkr_fills(ib: IB):
    fills = await ib.reqExecutionsAsync()

    rows = []

    for f in fills:
        c = f.contract
        e = f.execution
        cr = f.commissionReport

        rows.append({
            "time": f.time,
            "symbol": c.symbol,
            "exchange": e.exchange,
            "side": e.side,  # BOT / SLD
            "action": "BUY" if e.side == "BOT" else "SELL",
            "shares": float(e.shares),
            "price": float(e.price),
            "commission": cr.commission if cr else 0.0,
            "currency": c.currency,
            "execId": e.execId,
            "orderId": e.orderId,
        })

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values("time").reset_index(drop=True)

    return df


# --------------------------------------------------
# 2) FIFO PNL
# --------------------------------------------------
def calculate_fifo_pnl(df):
    lots = defaultdict(deque)
    realized = []
    open_lots = []

    for _, row in df.iterrows():
        symbol = row["symbol"]
        qty = row["shares"]
        price = row["price"]
        commission = row["commission"]
        side = row["side"]

        if side == "BOT":  # BUY
            lots[symbol].append({
                "qty": qty,
                "price": price,
                "time": row["time"],
                "commission": commission
            })

        elif side == "SLD":  # SELL
            sell_qty = qty

            while sell_qty > 0 and lots[symbol]:
                lot = lots[symbol][0]

                matched = min(sell_qty, lot["qty"])

                pnl = (price - lot["price"]) * matched

                realized.append({
                    "symbol": symbol,
                    "qty": matched,

                    "buy_price": lot["price"],
                    "buy_time": lot["time"],

                    "sell_price": price,
                    "sell_time": row["time"],

                    "pnl": pnl,
                    "pct": ((price - lot["price"]) / lot["price"]) * 100,
                })

                lot["qty"] -= matched
                sell_qty -= matched

                if lot["qty"] == 0:
                    lots[symbol].popleft()

    # açık lotlar
    for symbol, q in lots.items():
        for lot in q:
            open_lots.append({
                "symbol": symbol,
                "qty": lot["qty"],
                "buy_price": lot["price"],
                "buy_time": lot["time"]
            })

    return pd.DataFrame(realized), pd.DataFrame(open_lots)


# --------------------------------------------------
# 3) ANA RAPOR
# --------------------------------------------------
async def ibkr_trade_report_df(ib: IB):

    executions_df = await get_all_ibkr_fills(ib)

    realized_df, open_df = calculate_fifo_pnl(executions_df)

    # summary
    if not realized_df.empty:
        summary_df = realized_df.groupby("symbol").agg(
            total_qty=("qty", "sum"),
            total_pnl=("pnl", "sum"),
            avg_pct=("pct", "mean")
        ).reset_index()
    else:
        summary_df = pd.DataFrame()

    return {
        "executions": executions_df,
        "realized": realized_df,
        "open": open_df,
        "summary": summary_df
    }


report = await ibkr_trade_report_df(ib)

executions_df = report["executions"]
realized_df = report["realized"]
open_df = report["open"]
summary_df = report["summary"]


print("\n--- ALL EXECUTIONS ---")
display(executions_df)

print("\n--- REALIZED PNL ---")
display(realized_df)

print("\n--- OPEN POSITIONS ---")
display(open_df)

print("\n--- SUMMARY ---")
display(summary_df)


--- ALL EXECUTIONS ---


,time,symbol,exchange,side,action,shares,price,commission,currency,execId,orderId
0,2026-04-23 18:07:49+00:00,ISRG,NASDAQ,BOT,BUY,1.0,481.21,0.0,USD,00025b45.69ee175c.01.01,27
1,2026-04-23 18:07:52+00:00,COIN,NASDAQ,BOT,BUY,1.0,199.32,0.0,USD,00025b44.69eb0c3b.01.01,31
2,2026-04-23 18:07:56+00:00,MSFT,NASDAQ,BOT,BUY,1.0,415.22,0.0,USD,00025b45.69ee1793.01.01,35
3,2026-04-23 18:08:03+00:00,MOH,NYSE,BOT,BUY,1.0,169.40,0.0,USD,00025b45.69ee17ab.01.01,39
4,2026-04-23 18:08:06+00:00,OKLO,NYSE,BOT,BUY,1.0,76.95,0.0,USD,00025b49.69ec1adb.01.01,43
5,2026-04-23 18:24:37+00:00,ISRG,IBKRATS,SLD,SELL,1.0,481.71,0.0,USD,0000dc8f.6a3cd735.01.01,0
6,2026-04-23 18:46:41+00:00,AAPL,ARCA,BOT,BUY,1.0,273.79,0.0,USD,0000e0d5.69eb67a4.01.01,4
7,2026-04-23 18:46:53+00:00,AAPL,IEX,BOT,BUY,1.0,273.80,0.0,USD,0000e0d5.69eb67d7.01.01,4
8,2026-04-23 18:48:46+00:00,AAPL,IEX,BOT,BUY,1.0,274.08,0.0,USD,0000e0d5.69eb696b.01.01,4
9,2026-04-23 18:50:01+00:00,AAPL,NASDAQ,BOT,BUY,1.0,273.97,0.0,USD,0000e0d5.69eb6a7d.01.01,4



--- REALIZED PNL ---


,symbol,qty,buy_price,buy_time,sell_price,sell_time,pnl,pct
0,ISRG,1.0,481.21,2026-04-23 18:07:49+00:00,481.71,2026-04-23 18:24:37+00:00,0.50,0.103905
1,AAPL,1.0,273.79,2026-04-23 18:46:41+00:00,274.03,2026-04-23 18:51:56+00:00,0.24,0.087658
2,AAPL,1.0,273.80,2026-04-23 18:46:53+00:00,274.03,2026-04-23 18:51:56+00:00,0.23,0.084003
3,AAPL,1.0,274.08,2026-04-23 18:48:46+00:00,274.03,2026-04-23 18:51:56+00:00,-0.05,-0.018243
4,AAPL,1.0,273.97,2026-04-23 18:50:01+00:00,274.03,2026-04-23 18:51:56+00:00,0.06,0.021900
5,AAPL,1.0,273.94,2026-04-23 18:50:12+00:00,274.03,2026-04-23 18:51:56+00:00,0.09,0.032854
6,AAPL,1.0,273.98,2026-04-23 18:50:29+00:00,274.03,2026-04-23 18:51:56+00:00,0.05,0.018250
7,NVDA,1.0,198.90,2026-04-23 18:53:05+00:00,199.13,2026-04-23 19:22:58+00:00,0.23,0.115636
8,COIN,1.0,199.32,2026-04-23 18:07:52+00:00,197.88,2026-04-23 19:29:26+00:00,-1.44,-0.722456
9,AAPL,1.0,274.10,2026-04-23 18:52:19+00:00,274.14,2026-04-23 19:30:00+00:00,0.04,0.014593



--- OPEN POSITIONS ---


,symbol,qty,buy_price,buy_time
0,MSFT,1.0,415.22,2026-04-23 18:07:56+00:00
1,MSFT,1.0,416.35,2026-04-23 19:15:17+00:00
2,MSFT,1.0,416.67,2026-04-23 19:16:45+00:00
3,MOH,1.0,169.40,2026-04-23 18:08:03+00:00
4,OKLO,1.0,76.95,2026-04-23 18:08:06+00:00
5,GRAB,1.0,3.95,2026-04-23 18:56:32+00:00



--- SUMMARY ---


,symbol,total_qty,total_pnl,avg_pct
0,AAPL,7.0,0.66,0.034431
1,COIN,1.0,-1.44,-0.722456
2,ISRG,1.0,0.50,0.103905
3,NVDA,1.0,0.23,0.115636



--- ALL EXECUTIONS ---


,time,symbol,exchange,side,action,shares,price,commission,currency,execId,orderId
0,2026-04-23 18:07:49+00:00,ISRG,NASDAQ,BOT,BUY,1.0,481.21,0.0,USD,00025b45.69ee175c.01.01,27
1,2026-04-23 18:07:52+00:00,COIN,NASDAQ,BOT,BUY,1.0,199.32,0.0,USD,00025b44.69eb0c3b.01.01,31
2,2026-04-23 18:07:56+00:00,MSFT,NASDAQ,BOT,BUY,1.0,415.22,0.0,USD,00025b45.69ee1793.01.01,35
3,2026-04-23 18:08:03+00:00,MOH,NYSE,BOT,BUY,1.0,169.40,0.0,USD,00025b45.69ee17ab.01.01,39
4,2026-04-23 18:08:06+00:00,OKLO,NYSE,BOT,BUY,1.0,76.95,0.0,USD,00025b49.69ec1adb.01.01,43
5,2026-04-23 18:24:37+00:00,ISRG,IBKRATS,SLD,SELL,1.0,481.71,0.0,USD,0000dc8f.6a3cd735.01.01,0
6,2026-04-23 18:46:41+00:00,AAPL,ARCA,BOT,BUY,1.0,273.79,0.0,USD,0000e0d5.69eb67a4.01.01,4
7,2026-04-23 18:46:53+00:00,AAPL,IEX,BOT,BUY,1.0,273.80,0.0,USD,0000e0d5.69eb67d7.01.01,4
8,2026-04-23 18:48:46+00:00,AAPL,IEX,BOT,BUY,1.0,274.08,0.0,USD,0000e0d5.69eb696b.01.01,4
9,2026-04-23 18:50:01+00:00,AAPL,NASDAQ,BOT,BUY,1.0,273.97,0.0,USD,0000e0d5.69eb6a7d.01.01,4



--- REALIZED PNL ---


,symbol,qty,buy_price,buy_time,sell_price,sell_time,pnl,pct
0,ISRG,1.0,481.21,2026-04-23 18:07:49+00:00,481.71,2026-04-23 18:24:37+00:00,0.50,0.103905
1,AAPL,1.0,273.79,2026-04-23 18:46:41+00:00,274.03,2026-04-23 18:51:56+00:00,0.24,0.087658
2,AAPL,1.0,273.80,2026-04-23 18:46:53+00:00,274.03,2026-04-23 18:51:56+00:00,0.23,0.084003
3,AAPL,1.0,274.08,2026-04-23 18:48:46+00:00,274.03,2026-04-23 18:51:56+00:00,-0.05,-0.018243
4,AAPL,1.0,273.97,2026-04-23 18:50:01+00:00,274.03,2026-04-23 18:51:56+00:00,0.06,0.021900
5,AAPL,1.0,273.94,2026-04-23 18:50:12+00:00,274.03,2026-04-23 18:51:56+00:00,0.09,0.032854
6,AAPL,1.0,273.98,2026-04-23 18:50:29+00:00,274.03,2026-04-23 18:51:56+00:00,0.05,0.018250
7,NVDA,1.0,198.90,2026-04-23 18:53:05+00:00,199.13,2026-04-23 19:22:58+00:00,0.23,0.115636
8,COIN,1.0,199.32,2026-04-23 18:07:52+00:00,197.88,2026-04-23 19:29:26+00:00,-1.44,-0.722456
9,AAPL,1.0,274.10,2026-04-23 18:52:19+00:00,274.14,2026-04-23 19:30:00+00:00,0.04,0.014593



--- OPEN POSITIONS ---


,symbol,qty,buy_price,buy_time
0,MSFT,1.0,415.22,2026-04-23 18:07:56+00:00
1,MSFT,1.0,416.35,2026-04-23 19:15:17+00:00
2,MSFT,1.0,416.67,2026-04-23 19:16:45+00:00
3,MOH,1.0,169.40,2026-04-23 18:08:03+00:00
4,OKLO,1.0,76.95,2026-04-23 18:08:06+00:00
5,GRAB,1.0,3.95,2026-04-23 18:56:32+00:00



--- SUMMARY ---


,symbol,total_qty,total_pnl,avg_pct
0,AAPL,7.0,0.66,0.034431
1,COIN,1.0,-1.44,-0.722456
2,ISRG,1.0,0.50,0.103905
3,NVDA,1.0,0.23,0.115636
